In [1]:
import pandas as pd
import numpy as np 
import scanpy as sc 
import os
import scrublet as scr 
import glob

In [2]:
study_files = {
    "Dong2020_Group1": {
        "matrix": "Data_Dong2020_Neuroendocrine/Group1/Exp_data_UMIcounts1.mtx",
        "genes":  "Data_Dong2020_Neuroendocrine/Group1/Genes1.txt",
        "cells":  "Data_Dong2020_Neuroendocrine/Group1/Cells1.csv",
        "samples": "Data_Dong2020_Neuroendocrine/Samples.csv",
        "units": "UMI"
    },
    "Dong2020_Group2": {
        "matrix": "Data_Dong2020_Neuroendocrine/Group2/Exp_data_UMIcounts2.mtx",
        "genes":  "Data_Dong2020_Neuroendocrine/Group2/Genes2.txt",
        "cells":  "Data_Dong2020_Neuroendocrine/Group2/Cells2.csv",
        "samples": "Data_Dong2020_Neuroendocrine/Samples.csv",
        "units": "UMI"
    },
    "Jansky2021": {
        "matrix": "Data_Jansky2021_Neuroendocrine/Exp_data_UMIcounts.mtx",
        "genes":  "Data_Jansky2021_Neuroendocrine/Genes.txt",
        "cells":  "Data_Jansky2021_Neuroendocrine/Cells.csv",
        "samples": "Data_Jansky2021_Neuroendocrine/Samples.csv",
        "units": "UMI"
    },
    "Kildisiute2021_10X": {
        "matrix": "Data_Kildisiute2021_Neuroendocrine/10X/Exp_data_UMIcounts.mtx",
        "genes":  "Data_Kildisiute2021_Neuroendocrine/10X/Genes.txt",
        "cells":  "Data_Kildisiute2021_Neuroendocrine/10X/Cells.csv",
        "samples": "Data_Kildisiute2021_Neuroendocrine/Samples.csv",
        "units": "UMI"
    },
    "Kildisiute2021_CEL-seq2": {
        "matrix": "Data_Kildisiute2021_Neuroendocrine/CEL-seq2/Exp_data_TPM.mtx",
        "genes":  "Data_Kildisiute2021_Neuroendocrine/CEL-seq2/Genes.txt",
        "cells":  "Data_Kildisiute2021_Neuroendocrine/CEL-seq2/Cells.csv",
        "samples": "Data_Kildisiute2021_Neuroendocrine/Samples.csv",
        "units": "TPM"
    },
    "Rao2020": {
        "matrix": "Data_Rao2020_Neuroendocrine/Exp_data_UMIcounts.mtx",
        "genes":  "Data_Rao2020_Neuroendocrine/Genes.txt",
        "cells":  "Data_Rao2020_Neuroendocrine/Cells.csv",
        "samples": "Data_Rao2020_Neuroendocrine/Samples.csv",
        "units": "UMI"
    }
}

In [3]:
study_name = 'Dong2020_Group1'
paths = study_files[study_name]

# matrix
adata = sc.read_mtx(paths['matrix']).T

genes = pd.read_csv(paths["genes"], header=None, sep="\t").iloc[:, 0].tolist()
adata.var_names = genes
adata.var_names_make_unique()

cells = pd.read_csv(paths["cells"], index_col=0)
cells.index = cells.index.astype(str)
adata.obs_names = cells.index
adata.obs = adata.obs.join(cells, how="left")

adata.obs["study"] = study_name
print(f"Loaded: {adata.n_obs:,} cells × {adata.n_vars:,} genes")

# drop NaN
CRITICAL_COLS = ['cell_type', 'sample']
EXTRA_COLS = ['cancer_type', 'disease']
all_cols_to_check = [c for c in CRITICAL_COLS + EXTRA_COLS if c in adata.obs.columns]
drop_mask = pd.Series(False, index=adata.obs.index)

for col in all_cols_to_check:
    vals = adata.obs[col]
    nan_mask = vals.isna()
    if nan_mask.any():
        print(f"   {col}: {nan_mask.sum()} NaN")
        drop_mask |= nan_mask
    if vals.dtype == object:
        str_vals = vals.astype(str).str.strip().str.lower()
        suspect = str_vals.isin(['unknown', 'na', 'nan', 'none', ''])
        if suspect.any():
            print(f"   {col}: {suspect.sum()} suspicious")
            drop_mask |= suspect

print(f"Flagged {drop_mask.sum()} cells")
if drop_mask.any():
    adata = adata[~drop_mask, :].copy()
    print(f"After metadata drop: {adata.n_obs:,} cells")

# type col should exist
if 'cancer_type' not in adata.obs.columns:
    samples = pd.read_csv(paths["samples"], index_col=0)
    if 'cancer_type' in samples.columns:
        adata.obs['cancer_type'] = adata.obs['sample'].map(samples['cancer_type'])
    else:
        adata.obs['cancer_type'] = 'Neuroendocrine'   # PER STUDY CHANGE
    print(f"   Added cancer_type column. Unique values: {adata.obs['cancer_type'].unique()}")

# gene filter
sc.pp.calculate_qc_metrics(adata, inplace=True)
keep = (adata.obs["n_genes_by_counts"] >= 200) & (adata.obs["n_genes_by_counts"] <= 6000)
adata = adata[keep, :].copy()
print(f"After gene filter: {adata.n_obs:,} cells")

# doublet removal
counts = adata.X.toarray() if hasattr(adata.X, 'toarray') else adata.X
scrub = scr.Scrublet(counts, expected_doublet_rate=0.05)
doublet_scores, predicted_doublets = scrub.scrub_doublets()
adata.obs["doublet_score"] = doublet_scores
adata.obs["predicted_doublet"] = predicted_doublets
adata = adata[~adata.obs["predicted_doublet"], :].copy()
print(f"After doublet removal: {adata.n_obs:,} cells")

# mitochondria filter
adata.var['mt'] = adata.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)
keep_mt = adata.obs['pct_counts_mt'] <= 15
adata = adata[keep_mt, :].copy()
print(f"After MT filter: {adata.n_obs:,} cells")

# normalization
unit = study_files[study_name]["units"]
if unit == "TPM":
    sc.pp.log1p(adata)
else:
    sc.pp.normalize_total(adata, target_sum=1e6)   # CPM
    sc.pp.log1p(adata)
print(f"Normalised ({unit}) – max value {adata.X.max():.2f}")

# convert object cols and save
for col in adata.obs.columns:
    if adata.obs[col].dtype == 'object':
        adata.obs[col] = adata.obs[col].astype(str)

out_file = f"{study_name}.h5ad"
adata.write_h5ad(out_file)

Loaded: 26,433 cells × 12,595 genes
   cell_type: 2270 NaN
   cell_type: 2270 suspicious
Flagged 2270 cells
After metadata drop: 24,163 cells
   Added cancer_type column. Unique values: ['Neuroblastoma']
After gene filter: 24,079 cells
Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.53
Detected doublet rate = 0.0%
Estimated detectable doublet fraction = 47.2%
Overall doublet rate:
	Expected   = 5.0%
	Estimated  = 0.1%
Elapsed time: 24.5 seconds
After doublet removal: 24,069 cells
After MT filter: 24,069 cells
Normalised (UMI) – max value 12.34


In [4]:
study_name = 'Dong2020_Group2'
paths = study_files[study_name]

# matrix
adata = sc.read_mtx(paths['matrix']).T

genes = pd.read_csv(paths["genes"], header=None, sep="\t").iloc[:, 0].tolist()
adata.var_names = genes
adata.var_names_make_unique()

cells = pd.read_csv(paths["cells"], index_col=0)
cells.index = cells.index.astype(str)
adata.obs_names = cells.index
adata.obs = adata.obs.join(cells, how="left")

adata.obs["study"] = study_name
print(f"Loaded: {adata.n_obs:,} cells × {adata.n_vars:,} genes")

# drop NaN
CRITICAL_COLS = ['cell_type', 'sample']
EXTRA_COLS = ['cancer_type', 'disease']
all_cols_to_check = [c for c in CRITICAL_COLS + EXTRA_COLS if c in adata.obs.columns]
drop_mask = pd.Series(False, index=adata.obs.index)

for col in all_cols_to_check:
    vals = adata.obs[col]
    nan_mask = vals.isna()
    if nan_mask.any():
        print(f"   {col}: {nan_mask.sum()} NaN")
        drop_mask |= nan_mask
    if vals.dtype == object:
        str_vals = vals.astype(str).str.strip().str.lower()
        suspect = str_vals.isin(['unknown', 'na', 'nan', 'none', ''])
        if suspect.any():
            print(f"   {col}: {suspect.sum()} suspicious")
            drop_mask |= suspect

print(f"Flagged {drop_mask.sum()} cells")
if drop_mask.any():
    adata = adata[~drop_mask, :].copy()
    print(f"After metadata drop: {adata.n_obs:,} cells")

# type col should exist
if 'cancer_type' not in adata.obs.columns:
    samples = pd.read_csv(paths["samples"], index_col=0)
    if 'cancer_type' in samples.columns:
        adata.obs['cancer_type'] = adata.obs['sample'].map(samples['cancer_type'])
    else:
        adata.obs['cancer_type'] = 'Neuroendocrine'   # PER STUDY CHANGE
    print(f"   Added cancer_type column. Unique values: {adata.obs['cancer_type'].unique()}")

# gene filter
sc.pp.calculate_qc_metrics(adata, inplace=True)
keep = (adata.obs["n_genes_by_counts"] >= 200) & (adata.obs["n_genes_by_counts"] <= 6000)
adata = adata[keep, :].copy()
print(f"After gene filter: {adata.n_obs:,} cells")

# doublet removal
counts = adata.X.toarray() if hasattr(adata.X, 'toarray') else adata.X
scrub = scr.Scrublet(counts, expected_doublet_rate=0.05)
doublet_scores, predicted_doublets = scrub.scrub_doublets()
adata.obs["doublet_score"] = doublet_scores
adata.obs["predicted_doublet"] = predicted_doublets
adata = adata[~adata.obs["predicted_doublet"], :].copy()
print(f"After doublet removal: {adata.n_obs:,} cells")

# mitochondria filter
adata.var['mt'] = adata.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)
keep_mt = adata.obs['pct_counts_mt'] <= 15
adata = adata[keep_mt, :].copy()
print(f"After MT filter: {adata.n_obs:,} cells")

# normalization
unit = study_files[study_name]["units"]
if unit == "TPM":
    sc.pp.log1p(adata)
else:
    sc.pp.normalize_total(adata, target_sum=1e6)   # CPM
    sc.pp.log1p(adata)
print(f"Normalised ({unit}) – max value {adata.X.max():.2f}")

# convert object cols and save
for col in adata.obs.columns:
    if adata.obs[col].dtype == 'object':
        adata.obs[col] = adata.obs[col].astype(str)

out_file = f"{study_name}.h5ad"
adata.write_h5ad(out_file)

Loaded: 28,757 cells × 33,490 genes
   cell_type: 3887 NaN
   cell_type: 3887 suspicious
Flagged 3887 cells
After metadata drop: 24,870 cells
   Added cancer_type column. Unique values: ['Neuroblastoma']
After gene filter: 24,634 cells
Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.56
Detected doublet rate = 0.0%
Estimated detectable doublet fraction = 31.2%
Overall doublet rate:
	Expected   = 5.0%
	Estimated  = 0.1%
Elapsed time: 29.9 seconds
After doublet removal: 24,627 cells
After MT filter: 24,627 cells
Normalised (UMI) – max value 12.20


In [5]:
study_name = 'Jansky2021'
paths = study_files[study_name]

# matrix
adata = sc.read_mtx(paths['matrix']).T

genes = pd.read_csv(paths["genes"], header=None, sep="\t").iloc[:, 0].tolist()
adata.var_names = genes
adata.var_names_make_unique()

cells = pd.read_csv(paths["cells"], index_col=0)
cells.index = cells.index.astype(str)
adata.obs_names = cells.index
adata.obs = adata.obs.join(cells, how="left")

adata.obs["study"] = study_name
print(f"Loaded: {adata.n_obs:,} cells × {adata.n_vars:,} genes")

# drop NaN
CRITICAL_COLS = ['cell_type', 'sample']
EXTRA_COLS = ['cancer_type', 'disease']
all_cols_to_check = [c for c in CRITICAL_COLS + EXTRA_COLS if c in adata.obs.columns]
drop_mask = pd.Series(False, index=adata.obs.index)

for col in all_cols_to_check:
    vals = adata.obs[col]
    nan_mask = vals.isna()
    if nan_mask.any():
        print(f"   {col}: {nan_mask.sum()} NaN")
        drop_mask |= nan_mask
    if vals.dtype == object:
        str_vals = vals.astype(str).str.strip().str.lower()
        suspect = str_vals.isin(['unknown', 'na', 'nan', 'none', ''])
        if suspect.any():
            print(f"   {col}: {suspect.sum()} suspicious")
            drop_mask |= suspect

print(f"Flagged {drop_mask.sum()} cells")
if drop_mask.any():
    adata = adata[~drop_mask, :].copy()
    print(f"After metadata drop: {adata.n_obs:,} cells")

# type col should exist
if 'cancer_type' not in adata.obs.columns:
    samples = pd.read_csv(paths["samples"], index_col=0)
    if 'cancer_type' in samples.columns:
        adata.obs['cancer_type'] = adata.obs['sample'].map(samples['cancer_type'])
    else:
        adata.obs['cancer_type'] = 'Neuroendocrine'   # PER STUDY CHANGE
    print(f"   Added cancer_type column. Unique values: {adata.obs['cancer_type'].unique()}")

# gene filter
sc.pp.calculate_qc_metrics(adata, inplace=True)
keep = (adata.obs["n_genes_by_counts"] >= 200) & (adata.obs["n_genes_by_counts"] <= 6000)
adata = adata[keep, :].copy()
print(f"After gene filter: {adata.n_obs:,} cells")

# doublet removal
counts = adata.X.toarray() if hasattr(adata.X, 'toarray') else adata.X
scrub = scr.Scrublet(counts, expected_doublet_rate=0.05)
doublet_scores, predicted_doublets = scrub.scrub_doublets()
adata.obs["doublet_score"] = doublet_scores
adata.obs["predicted_doublet"] = predicted_doublets
adata = adata[~adata.obs["predicted_doublet"], :].copy()
print(f"After doublet removal: {adata.n_obs:,} cells")

# mitochondria filter
adata.var['mt'] = adata.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)
keep_mt = adata.obs['pct_counts_mt'] <= 15
adata = adata[keep_mt, :].copy()
print(f"After MT filter: {adata.n_obs:,} cells")

# normalization
unit = study_files[study_name]["units"]
if unit == "TPM":
    sc.pp.log1p(adata)
else:
    sc.pp.normalize_total(adata, target_sum=1e6)   # CPM
    sc.pp.log1p(adata)
print(f"Normalised ({unit}) – max value {adata.X.max():.2f}")

# convert object cols and save
for col in adata.obs.columns:
    if adata.obs[col].dtype == 'object':
        adata.obs[col] = adata.obs[col].astype(str)

out_file = f"{study_name}.h5ad"
adata.write_h5ad(out_file)

Loaded: 64,769 cells × 26,344 genes
   cell_type: 34579 NaN
   cell_type: 34579 suspicious
Flagged 34579 cells
After metadata drop: 30,190 cells
   Added cancer_type column. Unique values: ['Neuroblastoma']
After gene filter: 30,181 cells
Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.49
Detected doublet rate = 0.2%
Estimated detectable doublet fraction = 40.3%
Overall doublet rate:
	Expected   = 5.0%
	Estimated  = 0.4%
Elapsed time: 37.6 seconds
After doublet removal: 30,128 cells
After MT filter: 30,128 cells
Normalised (UMI) – max value 12.73


In [6]:
study_name = 'Kildisiute2021_10X'
paths = study_files[study_name]

# matrix
adata = sc.read_mtx(paths['matrix']).T

genes = pd.read_csv(paths["genes"], header=None, sep="\t").iloc[:, 0].tolist()
adata.var_names = genes
adata.var_names_make_unique()

cells = pd.read_csv(paths["cells"], index_col=0)
cells.index = cells.index.astype(str)
adata.obs_names = cells.index
adata.obs = adata.obs.join(cells, how="left")

adata.obs["study"] = study_name
print(f"Loaded: {adata.n_obs:,} cells × {adata.n_vars:,} genes")

# drop NaN
CRITICAL_COLS = ['cell_type', 'sample']
EXTRA_COLS = ['cancer_type', 'disease']
all_cols_to_check = [c for c in CRITICAL_COLS + EXTRA_COLS if c in adata.obs.columns]
drop_mask = pd.Series(False, index=adata.obs.index)

for col in all_cols_to_check:
    vals = adata.obs[col]
    nan_mask = vals.isna()
    if nan_mask.any():
        print(f"   {col}: {nan_mask.sum()} NaN")
        drop_mask |= nan_mask
    if vals.dtype == object:
        str_vals = vals.astype(str).str.strip().str.lower()
        suspect = str_vals.isin(['unknown', 'na', 'nan', 'none', ''])
        if suspect.any():
            print(f"   {col}: {suspect.sum()} suspicious")
            drop_mask |= suspect

print(f"Flagged {drop_mask.sum()} cells")
if drop_mask.any():
    adata = adata[~drop_mask, :].copy()
    print(f"After metadata drop: {adata.n_obs:,} cells")

# type col should exist
if 'cancer_type' not in adata.obs.columns:
    samples = pd.read_csv(paths["samples"], index_col=0)
    if 'cancer_type' in samples.columns:
        adata.obs['cancer_type'] = adata.obs['sample'].map(samples['cancer_type'])
    else:
        adata.obs['cancer_type'] = 'Neuroendocrine'   # PER STUDY CHANGE
    print(f"   Added cancer_type column. Unique values: {adata.obs['cancer_type'].unique()}")

# gene filter
sc.pp.calculate_qc_metrics(adata, inplace=True)
keep = (adata.obs["n_genes_by_counts"] >= 200) & (adata.obs["n_genes_by_counts"] <= 6000)
adata = adata[keep, :].copy()
print(f"After gene filter: {adata.n_obs:,} cells")

# doublet removal
counts = adata.X.toarray() if hasattr(adata.X, 'toarray') else adata.X
scrub = scr.Scrublet(counts, expected_doublet_rate=0.05)
doublet_scores, predicted_doublets = scrub.scrub_doublets()
adata.obs["doublet_score"] = doublet_scores
adata.obs["predicted_doublet"] = predicted_doublets
adata = adata[~adata.obs["predicted_doublet"], :].copy()
print(f"After doublet removal: {adata.n_obs:,} cells")

# mitochondria filter
adata.var['mt'] = adata.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)
keep_mt = adata.obs['pct_counts_mt'] <= 15
adata = adata[keep_mt, :].copy()
print(f"After MT filter: {adata.n_obs:,} cells")

# normalization
unit = study_files[study_name]["units"]
if unit == "TPM":
    sc.pp.log1p(adata)
else:
    sc.pp.normalize_total(adata, target_sum=1e6)   # CPM
    sc.pp.log1p(adata)
print(f"Normalised ({unit}) – max value {adata.X.max():.2f}")

# convert object cols and save
for col in adata.obs.columns:
    if adata.obs[col].dtype == 'object':
        adata.obs[col] = adata.obs[col].astype(str)

out_file = f"{study_name}.h5ad"
adata.write_h5ad(out_file)

Loaded: 6,442 cells × 33,428 genes
Flagged 0 cells
   Added cancer_type column. Unique values: ['Normal' 'Neuroblastoma']
After gene filter: 6,409 cells
Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.34
Detected doublet rate = 0.5%
Estimated detectable doublet fraction = 37.3%
Overall doublet rate:
	Expected   = 5.0%
	Estimated  = 1.3%
Elapsed time: 5.2 seconds
After doublet removal: 6,379 cells
After MT filter: 6,379 cells
Normalised (UMI) – max value 13.38


In [7]:
study_name = 'Kildisiute2021_CEL-seq2'
paths = study_files[study_name]

# matrix
adata = sc.read_mtx(paths['matrix']).T

genes = pd.read_csv(paths["genes"], header=None, sep="\t").iloc[:, 0].tolist()
adata.var_names = genes
adata.var_names_make_unique()

cells = pd.read_csv(paths["cells"], index_col=0)
cells.index = cells.index.astype(str)
adata.obs_names = cells.index
adata.obs = adata.obs.join(cells, how="left")

adata.obs["study"] = study_name
print(f"Loaded: {adata.n_obs:,} cells × {adata.n_vars:,} genes")

# drop NaN
CRITICAL_COLS = ['cell_type', 'sample']
EXTRA_COLS = ['cancer_type', 'disease']
all_cols_to_check = [c for c in CRITICAL_COLS + EXTRA_COLS if c in adata.obs.columns]
drop_mask = pd.Series(False, index=adata.obs.index)

for col in all_cols_to_check:
    vals = adata.obs[col]
    nan_mask = vals.isna()
    if nan_mask.any():
        print(f"   {col}: {nan_mask.sum()} NaN")
        drop_mask |= nan_mask
    if vals.dtype == object:
        str_vals = vals.astype(str).str.strip().str.lower()
        suspect = str_vals.isin(['unknown', 'na', 'nan', 'none', ''])
        if suspect.any():
            print(f"   {col}: {suspect.sum()} suspicious")
            drop_mask |= suspect

print(f"Flagged {drop_mask.sum()} cells")
if drop_mask.any():
    adata = adata[~drop_mask, :].copy()
    print(f"After metadata drop: {adata.n_obs:,} cells")

# type col should exist
if 'cancer_type' not in adata.obs.columns:
    samples = pd.read_csv(paths["samples"], index_col=0)
    if 'cancer_type' in samples.columns:
        adata.obs['cancer_type'] = adata.obs['sample'].map(samples['cancer_type'])
    else:
        adata.obs['cancer_type'] = 'Neuroendocrine'   # PER STUDY CHANGE
    print(f"   Added cancer_type column. Unique values: {adata.obs['cancer_type'].unique()}")

# gene filter
sc.pp.calculate_qc_metrics(adata, inplace=True)
keep = (adata.obs["n_genes_by_counts"] >= 200) & (adata.obs["n_genes_by_counts"] <= 6000)
adata = adata[keep, :].copy()
print(f"After gene filter: {adata.n_obs:,} cells")

# doublet removal
counts = adata.X.toarray() if hasattr(adata.X, 'toarray') else adata.X
scrub = scr.Scrublet(counts, expected_doublet_rate=0.05)
doublet_scores, predicted_doublets = scrub.scrub_doublets()
adata.obs["doublet_score"] = doublet_scores
adata.obs["predicted_doublet"] = predicted_doublets
adata = adata[~adata.obs["predicted_doublet"], :].copy()
print(f"After doublet removal: {adata.n_obs:,} cells")

# mitochondria filter
adata.var['mt'] = adata.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)
keep_mt = adata.obs['pct_counts_mt'] <= 15
adata = adata[keep_mt, :].copy()
print(f"After MT filter: {adata.n_obs:,} cells")

# normalization
unit = study_files[study_name]["units"]
if unit == "TPM":
    sc.pp.log1p(adata)
else:
    sc.pp.normalize_total(adata, target_sum=1e6)   # CPM
    sc.pp.log1p(adata)
print(f"Normalised ({unit}) – max value {adata.X.max():.2f}")

# convert object cols and save
for col in adata.obs.columns:
    if adata.obs[col].dtype == 'object':
        adata.obs[col] = adata.obs[col].astype(str)

out_file = f"{study_name}.h5ad"
adata.write_h5ad(out_file)

Loaded: 13,281 cells × 32,383 genes
   cell_type: 4517 NaN
   cell_type: 4517 suspicious
Flagged 4517 cells
After metadata drop: 8,764 cells
   Added cancer_type column. Unique values: ['Neuroblastoma']
After gene filter: 8,553 cells
Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.37
Detected doublet rate = 0.5%
Estimated detectable doublet fraction = 16.7%
Overall doublet rate:
	Expected   = 5.0%
	Estimated  = 2.9%
Elapsed time: 6.7 seconds
After doublet removal: 8,511 cells
After MT filter: 7,797 cells
Normalised (TPM) – max value 9.43


In [8]:
study_name = 'Rao2020'
paths = study_files[study_name]

# matrix
adata = sc.read_mtx(paths['matrix']).T

genes = pd.read_csv(paths["genes"], header=None, sep="\t").iloc[:, 0].tolist()
adata.var_names = genes
adata.var_names_make_unique()

cells = pd.read_csv(paths["cells"], index_col=0)
cells.index = cells.index.astype(str)
adata.obs_names = cells.index
adata.obs = adata.obs.join(cells, how="left")

adata.obs["study"] = study_name
print(f"Loaded: {adata.n_obs:,} cells × {adata.n_vars:,} genes")

# drop NaN
CRITICAL_COLS = ['cell_type', 'sample']
EXTRA_COLS = ['cancer_type', 'disease']
all_cols_to_check = [c for c in CRITICAL_COLS + EXTRA_COLS if c in adata.obs.columns]
drop_mask = pd.Series(False, index=adata.obs.index)

for col in all_cols_to_check:
    vals = adata.obs[col]
    nan_mask = vals.isna()
    if nan_mask.any():
        print(f"   {col}: {nan_mask.sum()} NaN")
        drop_mask |= nan_mask
    if vals.dtype == object:
        str_vals = vals.astype(str).str.strip().str.lower()
        suspect = str_vals.isin(['unknown', 'na', 'nan', 'none', ''])
        if suspect.any():
            print(f"   {col}: {suspect.sum()} suspicious")
            drop_mask |= suspect

print(f"Flagged {drop_mask.sum()} cells")
if drop_mask.any():
    adata = adata[~drop_mask, :].copy()
    print(f"After metadata drop: {adata.n_obs:,} cells")

# type col should exist
if 'cancer_type' not in adata.obs.columns:
    samples = pd.read_csv(paths["samples"], index_col=0)
    if 'cancer_type' in samples.columns:
        adata.obs['cancer_type'] = adata.obs['sample'].map(samples['cancer_type'])
    else:
        adata.obs['cancer_type'] = 'Neuroendocrine'   # PER STUDY CHANGE
    print(f"   Added cancer_type column. Unique values: {adata.obs['cancer_type'].unique()}")

# gene filter
sc.pp.calculate_qc_metrics(adata, inplace=True)
keep = (adata.obs["n_genes_by_counts"] >= 200) & (adata.obs["n_genes_by_counts"] <= 6000)
adata = adata[keep, :].copy()
print(f"After gene filter: {adata.n_obs:,} cells")

# doublet removal
counts = adata.X.toarray() if hasattr(adata.X, 'toarray') else adata.X
scrub = scr.Scrublet(counts, expected_doublet_rate=0.05)
doublet_scores, predicted_doublets = scrub.scrub_doublets()
adata.obs["doublet_score"] = doublet_scores
adata.obs["predicted_doublet"] = predicted_doublets
adata = adata[~adata.obs["predicted_doublet"], :].copy()
print(f"After doublet removal: {adata.n_obs:,} cells")

# mitochondria filter
adata.var['mt'] = adata.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)
keep_mt = adata.obs['pct_counts_mt'] <= 15
adata = adata[keep_mt, :].copy()
print(f"After MT filter: {adata.n_obs:,} cells")

# normalization
unit = study_files[study_name]["units"]
if unit == "TPM":
    sc.pp.log1p(adata)
else:
    sc.pp.normalize_total(adata, target_sum=1e6)   # CPM
    sc.pp.log1p(adata)
print(f"Normalised ({unit}) – max value {adata.X.max():.2f}")

# convert object cols and save
for col in adata.obs.columns:
    if adata.obs[col].dtype == 'object':
        adata.obs[col] = adata.obs[col].astype(str)

out_file = f"{study_name}.h5ad"
adata.write_h5ad(out_file)

Loaded: 4,739 cells × 25,006 genes
   cell_type: 1581 NaN
   cell_type: 1581 suspicious
Flagged 1581 cells
After metadata drop: 3,158 cells
   Added cancer_type column. Unique values: ['Neuroendocrine Tumor']
After gene filter: 3,145 cells
Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.48
Detected doublet rate = 0.3%
Estimated detectable doublet fraction = 4.9%
Overall doublet rate:
	Expected   = 5.0%
	Estimated  = 5.2%
Elapsed time: 1.4 seconds
After doublet removal: 3,137 cells
After MT filter: 2,839 cells
Normalised (UMI) – max value 13.07


In [9]:
INPUT_PATTERN = "*.h5ad"   

files = sorted(glob.glob(INPUT_PATTERN))

for f in files:
    adata = sc.read_h5ad(f)
    study_name = adata.obs['study'].iloc[0] if 'study' in adata.obs.columns else f
    print(f"{study_name}  –  {adata.n_obs:,} cells, {adata.n_vars:,} genes")
    
    for col in sorted(adata.obs.columns):
        if pd.api.types.is_numeric_dtype(adata.obs[col]):
            continue
        
        vals = adata.obs[col].dropna().unique()
        n_unique = len(vals)
        
        if n_unique <= 25:
            val_str = ', '.join(str(v) for v in sorted(vals))
        else:
            val_str = f"{n_unique} unique values – first 15: " + ', '.join(str(v) for v in sorted(vals)[:15])
        
        print(f" {col}: {val_str}")
    
    print()

Dong2020_Group1  –  24,069 cells, 12,595 genes
 cancer_type: Neuroblastoma
 cell_cycle_phase: G1/S, G2/M, Intermediate, Not cycling, nan
 cell_type: B_cell, Dendritic, Endothelial, Fibroblast, Malignant, Myeloid, Schwann, T_cell
 mp_assignment: CAF1, CAF7, CD4 - Cell Cycle, CD4 - Cytotoxic, CD8 - Cell Cycle, CD8 - Cytotoxic, Cell Cycle - G1/S, Cell Cycle - G2/M, Complement, Hypoxia, Interferon/MHC-II (II), Pericyte-like, Proteasomal degradation, Protein maturation, Stress, Stress (in vitro), Translation initiation, nan
 mp_top: 83 unique values – first 15: Alveolar, Astrocytes, CAF1, CAF10, CAF2, CAF3, CAF4, CAF5, CAF6, CAF7, CAF8, CAF9, CD4 - Cell Cycle, CD4 - Cytotoxic, CD4 - Dysfunction
 sample: Tumor_10, Tumor_19, Tumor_27, Tumor_34, Tumor_40, Tumor_44, Tumor_69, Tumor_71, Tumor_75, Tumor_92
 study: Dong2020_Group1

Dong2020_Group2  –  24,627 cells, 33,490 genes
 cancer_type: Neuroblastoma
 cell_cycle_phase: G1/S, G2/M, Intermediate, Not cycling
 cell_type: Malignant
 mp_assignment

In [10]:

INPUT_PATTERN = "*.h5ad"  
OUTPUT_FILE = "neuroendocrine.h5ad"

valid_prefixes = [
    'Dong2020_Group1', 'Dong2020_Group2',
    'Jansky2021',
    'Kildisiute2021_10X', 'Kildisiute2021_CEL-seq2',
    'Rao2020'
]
all_files = sorted(glob.glob(INPUT_PATTERN))
real_files = [f for f in all_files if any(f.startswith(p) for p in valid_prefixes)]
print(f"Found {len(real_files)} neuroendocrine study files (ignored {len(all_files)-len(real_files)} other files)\n")

def label_malignant(adata, study_name):
    if 'cell_type' in adata.obs.columns:
        ct = adata.obs['cell_type'].astype(str).str.lower().str.strip()
        if ct.str.contains('malignant', na=False).any():
            return ct.str.contains('malignant', na=False), "cell_type='Malignant'"
    

    return pd.Series(False, index=adata.obs.index), "all non‑malignant"
temp_files = []
total_malig = 0
total_non = 0

for f in real_files:
    adata = sc.read_h5ad(f)
    study_name = adata.obs['study'].iloc[0]
    
    malignant_mask, strategy = label_malignant(adata, study_name)
    adata.obs['is_malignant'] = malignant_mask.astype(int)
    
    malig_count = malignant_mask.sum()
    non_count = (~malignant_mask).sum()
    total_malig += malig_count
    total_non += non_count
    
    print(f"{study_name:<25s}  {adata.n_obs:>7,} cells  →  malignant: {malig_count:>6,}   non‑malignant: {non_count:>7,}   ({strategy})")
    
    temp_file = f"__temp_{study_name}.h5ad"
    adata.write_h5ad(temp_file)
    temp_files.append(temp_file)

print(f"\n{'='*60}")
print(f"TOTAL: {total_malig+total_non:>7,} cells  →  malignant: {total_malig:>6,}   non‑malignant: {total_non:>7,}")

print(f"\nMerging {len(temp_files)} studies with inner gene join...")
adatas = [sc.read_h5ad(tf) for tf in temp_files]
adata_all = adatas[0].concatenate(
    adatas[1:],
    batch_key='study',
    join='inner',
    index_unique='-'
)
print(f"Merged: {adata_all.n_obs:,} cells × {adata_all.n_vars:,} genes")

study_names_original = []
for tf in temp_files:
    ad = sc.read_h5ad(tf)
    study_names_original.append(ad.obs['study'].iloc[0])
study_map = {str(i): name for i, name in enumerate(study_names_original)}
adata_all.obs['study'] = adata_all.obs['study'].astype(str).map(study_map)
print(f"Study names restored: {sorted(adata_all.obs['study'].unique())}")


essential = ['sample', 'cell_type', 'cancer_type', 'study', 'is_malignant']
keep = [c for c in essential if c in adata_all.obs.columns]
adata_all.obs = adata_all.obs[keep]
print(f"Metadata retained: {list(adata_all.obs.columns)}")

print("\nCancer types:", sorted(adata_all.obs['cancer_type'].astype(str).unique()))
print("Malignant distribution:")
print(adata_all.obs['is_malignant'].value_counts())

# Drop NaN if any
for col in keep:
    n_nan = adata_all.obs[col].isna().sum()
    if n_nan > 0:
        print(f" {col}: {n_nan} NaN – dropping")
        adata_all = adata_all[~adata_all.obs[col].isna(), :].copy()

for col in adata_all.obs.columns:
    if adata_all.obs[col].dtype == 'object':
        adata_all.obs[col] = adata_all.obs[col].astype(str)

print(f"Final clean dataset: {adata_all.n_obs:,} cells × {adata_all.n_vars:,} genes")
adata_all.write_h5ad(OUTPUT_FILE)

# Clean up temp files
for tf in temp_files:
    os.remove(tf)
print("Temporary files cleaned up.")

Found 6 neuroendocrine study files (ignored 0 other files)

Dong2020_Group1             24,069 cells  →  malignant: 22,426   non‑malignant:   1,643   (cell_type='Malignant')
Dong2020_Group2             24,627 cells  →  malignant: 24,627   non‑malignant:       0   (cell_type='Malignant')
Jansky2021                  30,128 cells  →  malignant: 28,056   non‑malignant:   2,072   (cell_type='Malignant')
Kildisiute2021_10X           6,379 cells  →  malignant:  1,732   non‑malignant:   4,647   (cell_type='Malignant')
Kildisiute2021_CEL-seq2      7,797 cells  →  malignant:    601   non‑malignant:   7,196   (cell_type='Malignant')
Rao2020                      2,839 cells  →  malignant:  1,947   non‑malignant:     892   (cell_type='Malignant')

TOTAL:  95,839 cells  →  malignant: 79,389   non‑malignant:  16,450

Merging 6 studies with inner gene join...


/tmp/ipykernel_50285/23459471.py:49: FutureWarning: The method concatenate is deprecated and will be removed in the future. Use anndata.concat instead of AnnData.concatenate. See the tutorial for concat at: https://anndata.readthedocs.io/en/latest/concatenation.html
  adata_all = adatas[0].concatenate(


Merged: 95,839 cells × 11,057 genes
Study names restored: ['Dong2020_Group1', 'Dong2020_Group2', 'Jansky2021', 'Kildisiute2021_10X', 'Kildisiute2021_CEL-seq2', 'Rao2020']
Metadata retained: ['sample', 'cell_type', 'cancer_type', 'study', 'is_malignant']

Cancer types: ['Neuroblastoma', 'Neuroendocrine Tumor', 'Normal']
Malignant distribution:
is_malignant
1    79389
0    16450
Name: count, dtype: int64
Final clean dataset: 95,839 cells × 11,057 genes
Temporary files cleaned up.
